#### View data

In [122]:
def DisplayData(
    tgt_type: str = "",    # OS, VOL, TAB
    catalog: str = "",
    schema: str = "",
    volume: str = "",
    folder: str = "",
    oci_bucket: str = "",
    tbl_list: str = "",
    distype: str = "",     # count, data, lineage, version
    version: str = ""
):
    tables_df = spark.sql(f"SHOW TABLES IN {catalog}.{schema}")

    if tbl_list.strip() != "":
        tables_df = tables_df.filter(f"lower(tableName) in ({tbl_list})")

    # Handle accidental tuple input like ('count',)
    if isinstance(distype, tuple):
        distype = distype[0] if len(distype) > 0 else ""

    tgt = str(tgt_type).lower().strip()
    mode = str(distype).lower().strip()

    for r in tables_df.collect():
        table_name = r["tableName"]

        if tgt == "tab":
            full_name = f"{catalog}.{schema}.{table_name}"
            delta_ref = full_name
        elif tgt == "vol":
            full_name = f"/Volumes/{catalog}/{schema}/{volume}/{folder}/{table_name}"
            delta_ref = f"'{full_name}'"
        elif tgt == "os":
            full_name = f"{oci_bucket}/{folder}/{table_name}"
            delta_ref = f"'{full_name}'"
        else:
            raise ValueError(f"Invalid tgt_type: {tgt_type}")

        if mode == "lineage":
            lineage_df = spark.sql(f"DESCRIBE HISTORY {delta_ref}")
            print(f"Lineage for {full_name}")
            display(lineage_df)

        elif mode == "version":
            if tgt == "tab":
                df = spark.sql(
                    f"SELECT * FROM {full_name} VERSION AS OF {int(version)}"
                )
            else:
                df = (
                    spark.read.format("delta")
                    .option("versionAsOf", int(version))
                    .load(full_name)
                )

            print(f"Displaying version {version} for {full_name}")
            display(df)

        else:
            if tgt == "tab":
                df = spark.read.table(full_name)
            else:
                df = spark.read.format("delta").load(full_name)

            if mode == "count":
                print(f"{full_name} Record count = {df.count()}")
            elif mode == "data":
                display(df)

In [133]:
TGT_TYPE="os"
CATALOG="default"
SCHEMA="default"
VOLUME="psftos"
FOLDER="bronze"
OCI_BUCKET="oci://AIDPTEST@orasenatdpltintegration03"
TBL_LIST="'ps_payer','ps_provider'"
DISTYPE="data",
VERSION=1
DisplayData(
    tgt_type=TGT_TYPE,
    catalog=CATALOG,
    schema=SCHEMA,
    volume=VOLUME,
    folder=FOLDER,
    oci_bucket=OCI_BUCKET,
    tbl_list=TBL_LIST,
    distype=DISTYPE,
    version=VERSION
)